# IOAI — 2026 Mock Word Segmentation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-word-segmentation/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Compound Word Segmentation — 독일어 복합어 분할 (APOAI 2026 Mock / NOAI2025)

독일어 복합어(예: `Fußballspieler` → Fuß·ball·spieler)를 하위단어로 분할한다. **글자별 0/1 라벨**
(1 = 하위단어의 끝). `data/train.json`(94k, 라벨 있음)으로 학습하고 `data/test.json`(라벨 비어있음)을 채워
`submission.json`={word:[0/1,...]} 를 만든다. 점수 = 단어별 **분할경계 F1** 평균(경계 정확일치). GPU ≤10분.
힌트: Embedding + 딥러닝(BiLSTM).

In [ ]:
import json, numpy as np
train = json.load(open("data/train.json"))       # {word: [0/1,...]}
test  = json.load(open("data/test.json"))         # {word: []}  (라벨 비어있음)
test_words = list(test.keys())
print("train", len(train), "| test", len(test_words))

In [ ]:
# 베이스라인: 분할 안 함(단어 전체를 한 조각으로) — 마지막 글자만 경계
sub = {w: [0]*(len(w)-1) + [1] for w in test_words}
json.dump(sub, open("submission.json","w"))
print("saved submission.json", len(sub))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)